# Task 3 — No History, and No Single Truth

**Ali Raza** · Roll No: **su92-bsdsm-f23-018** · Section: **BSDS-7A**
Data Warehouse & BI — Week 1

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#9aa4b2", "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937", "xtick.color": "#4b5563", "ytick.color": "#4b5563",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.7,
    "axes.axisbelow": True, "font.size": 10,
})
INK, BLUE, RED, GREY = "#1f2937", "#2563eb", "#dc2626", "#9ca3af"
os.makedirs("charts", exist_ok=True)
NOTE = "DWBI Week 1 · synthetic course data"


## 1. No history — "Where did Ali's sales go?"

**Which of Inmon's four properties breaks:** **time-variant**. A warehouse keeps every fact as it was true *at the time*; the live table kept only the current truth, so when Ali's `Branch` was overwritten in March, last year's Lahore history was destroyed. Subject-oriented, integrated and non-volatile are all intact — the data is simply no longer *true for last year*.

**What the warehouse does differently:** it never overwrites. Ali gets **two rows** (a slowly changing dimension): Ali–Lahore valid Jan–Feb, Ali–Karachi valid from March. Last April's report joins each sale to the branch row that was valid *when the sale happened* — Lahore keeps its history, Karachi gets only March onwards.

**Another business situation:** a product's **price** changes. Overwrite the price and every old invoice suddenly "sold" at the new price — revenue history is rewritten. The warehouse keeps price-effective-dated rows instead.

## 2. No single truth — three departments, three numbers

**Plausible reasons (no one lying, no arithmetic error):**
- **Operations (Rs 661.65 m)** counts every order *shipped* last month — their job is fulfilment.
- **Finance (Rs 283.70 m)** counts only invoices *paid* last month — their job is cash.
- **Sales (Rs 584.75 m)** counts every order *booked* last month — their job is pipeline.

Same month, three different *definitions* of "sales". Each number is correct under its own filter.

**What a single staging area changes:** there is **one pipeline** that lands every order once, with one governed definition of "a sale" (e.g. shipped, net of returns). The three numbers don't disappear — they become three *views* over the same conformed data, and anyone can see exactly which filter produces which figure. Disagreement moves from "whose spreadsheet is right" to "which definition do we want".

**Who owns the definition:** a **data owner / governance group from the business** — not IT, and not any single department. Not each department, because three private definitions mean three "truths" and every cross-department decision becomes an argument about numbers instead of about the business.

## 3. One paragraph — "why build a warehouse at all?"

Because your systems already demonstrate the two failures: the live database **forgets** — overwrite Ali's branch and last year's report is wrong forever — and it **disagrees with itself** — three departments, three "sales" figures, all honest. A warehouse exists to fix exactly these two things: it keeps history (time-variant, non-volatile) so the past stays true, and it integrates everything through one staging area so there is one governed definition of each fact. The data you have is the raw material; the warehouse is what makes it *trustworthy*.

## The question to answer — "non-volatile", properly

**Better one-sentence definition:** once data is loaded into the warehouse it is **never updated in place** — corrections arrive as *new* rows, so history is append-only and the past can't be rewritten.

**One legitimate change:** fixing a **bad load** — if Tuesday night's ETL job loaded corrupted rows, the warehouse team corrects or removes *that load's* rows (with an audit note). That's repairing the pipeline's mistake, not rewriting business history.